In [2]:
import os
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv('config/db_config.env')

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

df = pd.read_sql('SELECT * FROM loans_clean', engine)
print(f'Shape: {df.shape}')
df.head()

Shape: (149986, 16)


,loan_id,serious_delinquency,revolving_utilization,age,times_30_59_days_late,times_60_89_days_late,times_90_days_late,monthly_income,debt_ratio,open_credit_lines,real_estate_loans,number_of_dependents,age_group,income_band,utilization_band,total_delinquency_events
0,1,1,0.7661,45,2,0,0,9120.0,0.8030,13,6,2,45_to_59,high,high,2
1,2,0,0.9572,40,0,0,0,2600.0,0.1219,4,0,1,30_to_44,low,critical,0
2,3,0,0.6582,38,1,0,1,3042.0,0.0851,2,0,0,30_to_44,medium,high,2
3,4,0,0.2338,30,0,0,0,3300.0,0.0360,5,0,0,30_to_44,medium,low,0
4,5,0,0.9072,49,1,0,0,63588.0,0.0249,7,1,0,45_to_59,very_high,critical,1


In [3]:
print(f"Total borrowers : {len(df):,}")
print(f"Total defaults  : {df['serious_delinquency'].sum():,}")
print(f"Default rate    : {df['serious_delinquency'].mean()*100:.2f}%")
df.describe().round(2)

Total borrowers : 149,986
Total defaults  : 10,025
Default rate    : 6.68%


,loan_id,serious_delinquency,revolving_utilization,age,times_30_59_days_late,times_60_89_days_late,times_90_days_late,monthly_income,debt_ratio,open_credit_lines,real_estate_loans,number_of_dependents,total_delinquency_events
count,149986.00,149986.00,149986.00,149986.00,149986.00,149986.00,149986.00,149986.00,149986.00,149986.00,149986.00,149986.00,149986.00
mean,75000.50,0.07,0.32,52.29,0.42,0.24,0.27,6418.68,2.26,8.45,1.02,0.74,0.93
std,43301.65,0.25,0.35,14.76,4.19,4.16,4.17,12890.96,3.84,5.15,1.13,1.11,12.47
min,1.00,0.00,0.00,21.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,37500.25,0.00,0.03,41.00,0.00,0.00,0.00,3903.25,0.18,5.00,0.00,0.00,0.00
50%,75000.50,0.00,0.15,52.00,0.00,0.00,0.00,5400.00,0.37,8.00,1.00,0.00,0.00
75%,112500.75,0.00,0.56,63.00,0.00,0.00,0.00,7400.00,0.87,11.00,2.00,1.00,0.00
max,150000.00,1.00,1.00,99.00,98.00,98.00,98.00,3008750.00,10.00,58.00,54.00,20.00,294.00


In [4]:
age_order = ['under_30', '30_to_44', '45_to_59', '60_plus']

age_df = (
    df.groupby('age_group')
    .agg(total=('serious_delinquency','count'),
         defaults=('serious_delinquency','sum'))
    .assign(default_rate=lambda x: (x.defaults/x.total*100).round(2))
    .reindex(age_order)
    .reset_index()
)

px.bar(age_df, x='default_rate', y='age_group',
       orientation='h',
       title='Default Rate by Age Group',
       color='default_rate',
       color_continuous_scale='RdYlGn_r')

In [5]:
def bucket(x):
    if x == 0: return '0_clean'
    if x == 1: return '1_event'
    if x == 2: return '2_events'
    if x <= 5: return '3_to_5_events'
    return '6_plus_events'

df['delinquency_bucket'] = df['total_delinquency_events'].apply(bucket)

bucket_order = ['0_clean','1_event','2_events','3_to_5_events','6_plus_events']

dlq_df = (
    df.groupby('delinquency_bucket')
    .agg(total=('serious_delinquency','count'),
         defaults=('serious_delinquency','sum'))
    .assign(default_rate=lambda x: (x.defaults/x.total*100).round(2))
    .reindex(bucket_order)
    .reset_index()
)

px.bar(dlq_df, x='delinquency_bucket', y='default_rate',
       title='Default Rate by Delinquency History',
       color='default_rate',
       color_continuous_scale='RdYlGn_r')

In [6]:
px.histogram(
    df, x='revolving_utilization',
    color=df['serious_delinquency'].astype(str),
    barmode='overlay', nbins=50, opacity=0.7,
    title='Utilization Distribution by Default Status',
    color_discrete_map={'0':'#2D7DD2','1':'#F85149'}
)

In [7]:
cols = ['serious_delinquency','revolving_utilization','age',
        'monthly_income','debt_ratio','total_delinquency_events']

px.imshow(df[cols].corr().round(2),
          title='Feature Correlation Matrix',
          color_continuous_scale='RdBu_r',
          zmin=-1, zmax=1, text_auto=True)